**COMANDO WSL2 — EXECUTAR FASE COMPLETA**

```bash
make fase1
```

**TÍTULO**

F1A — Atualização e ingestão das fontes do NINO26

**CONTEXTO**

A Fase 1 é responsável por baixar, atualizar, catalogar e auditar as fontes locais. Este notebook organiza OISST, ERA5, CHIRPS, UFS+GLORYS, CTD/WOD e as malhas IBGE de estados, regiões e biomas. Ele não trata nem interpreta os dados; essa função pertence à Fase 2 e às fases analíticas.

**MOTIVAÇÃO**

Hipótese operacional: a base local pode ser atualizada de forma retomável, mantendo procedência, cobertura real e falhas explícitas. O modo padrão é apenas diagnóstico; downloads só são iniciados quando `NINO26_UPDATE_DATA=1`.

**METODOLOGIA**

O notebook chama o pipeline incremental da Fase 1, que verifica arquivos existentes, baixa somente pendências, respeita a latência de cada fonte e executa a auditoria das malhas IBGE. Cada comando registra código de saída e horário.

**RESULTADOS ESPERADOS**

- plano de atualização por fonte;
- atualização retomável quando autorizada;
- auditoria IBGE de 27 UFs, 5 regiões e 6 biomas;
- inventário final com arquivos e datas locais;
- contrato de saída para a Fase 2, sem executar tratamentos da F2.

**CONFIGURAÇÃO OPERACIONAL**

Por segurança, a primeira execução apenas mostra o plano. Para atualizar de fato, defina `NINO26_UPDATE_DATA=1` no ambiente e execute novamente o notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
PYTHON = ROOT / '.venv' / 'Scripts' / 'python.exe'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)
EXECUTE_UPDATE = os.environ.get('NINO26_UPDATE_DATA', '0') == '1'
print(f'Raiz: {ROOT}')
print(f'Atualização autorizada: {EXECUTE_UPDATE}')
print(f'Início UTC: {datetime.now(timezone.utc).isoformat()}')

**PLANO E ATUALIZAÇÃO DAS FONTES**

In [ ]:
command = [str(PYTHON), 'scripts/run_full_download_pipeline.py']
if EXECUTE_UPDATE:
    command.extend(['--execute', '--stop-on-error'])
print('Comando:', ' '.join(command))
completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=False)
print(completed.stdout[-12000:])
if completed.stderr:
    print(completed.stderr[-6000:])
if completed.returncode:
    raise RuntimeError(f'Pipeline F1 terminou com código {completed.returncode}')

**AUDITORIA DAS MALHAS IBGE**

In [ ]:
audit_command = [str(PYTHON), 'scripts/audit_ibge_boundaries.py']
audit = subprocess.run(audit_command, cwd=ROOT, text=True, capture_output=True, check=False)
print(audit.stdout)
if audit.returncode:
    raise RuntimeError('A auditoria IBGE encontrou problemas.')
ibge_report = ROOT / 'data/processed/parquet/statistics/phase1_ibge_boundaries_audit.csv'
display(pd.read_csv(ibge_report))

**INVENTÁRIO LOCAL APÓS A ATUALIZAÇÃO**

In [ ]:
roots = [ROOT/'data/raw', ROOT/'data/interim', ROOT/'data/processed/zarr']
rows = []
for base in roots:
    if not base.exists():
        continue
    files = [p for p in base.rglob('*') if p.is_file()]
    rows.append({'arvore': str(base.relative_to(ROOT)), 'arquivos': len(files), 'ultima_modificacao_utc': max((datetime.fromtimestamp(p.stat().st_mtime, timezone.utc) for p in files), default=pd.NaT)})
inventory = pd.DataFrame(rows)
display(inventory)

**REFERÊNCIAS BIBLIOGRÁFICAS**

1. NOAA Climate Prediction Center. Oceanic Niño Index. https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php
2. NOAA OISST v2.1. https://www.ncei.noaa.gov/products/optimum-interpolation-sst
3. Climate Hazards Center. CHIRPS. https://www.chc.ucsb.edu/data/chirps
4. ECMWF. ERA5. https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5
5. IBGE. Geociências e malhas territoriais. https://www.ibge.gov.br/geociencias/